**Automatic Speech Recoginition: Speech To Text**

## Install Libraries

In [1]:
%pip install datasets transformers evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 11.6 MB/s eta 0:00:00


In [2]:
import torch
import datasets
import evaluate
import numpy as np
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union
from transformers import AutoModelForCTC, TrainingArguments, Trainer, Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor
from google.colab import userdata

## Load Dataset

[SPRINGLab/asr-task-data](https://huggingface.co/datasets/SPRINGLab/asr-task-data)

In [3]:
dataset = datasets.load_dataset("SPRINGLab/asr-task-data",token=userdata.get("HF_TOKEN"))

README.md:   0%|          | 0.00/356 [00:00<?, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/397M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/488M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/375M [00:00<?, ?B/s]

data/train-00003-of-00004.parquet:   0%|          | 0.00/335M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8000 [00:00<?, ? examples/s]

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['audio', 'text'],
        num_rows: 8000
    })
})

In [6]:
dataset['train'][0]

{'audio': <datasets.features._torchcodec.AudioDecoder at 0x7feee58e1130>,
 'text': 'block which lets get this one running so the simple part is just go over there and then'}

This function takes a batch of data as input. Each batch contains text samples. It concatenates all the text samples in the batch into a single string `all_text`. It then creates a list of unique characters `vocab` from this combined text and returns a dictionary with two keys: `vocab`, containing the list of unique characters, and `all_text`, containing the combined text.

In [7]:
def extract_all_chars(batch):
  all_text = " ".join(batch["text"])
  vocab = list(set(all_text))
  return {"vocab": [vocab], "all_text": [all_text]}

In [8]:
vocabs = dataset.map(extract_all_chars, batch_size=8)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

`vocab_list` is initialized to hold all characters found across the dataset.
We iterate over the vocabularies from each batch in the training set and extends `vocab_list` with the characters from each batch's vocabulary.

In [9]:
vocab_list = []

In [10]:
for v in vocabs["train"]["vocab"]:
  vocab_list.extend(v[0])

In [11]:
vocab_list

['l',
 'j',
 'w',
 'n',
 'd',
 'g',
 'h',
 'v',
 'k',
 'm',
 'a',
 'e',
 'u',
 'i',
 'o',
 'c',
 ' ',
 'b',
 'r',
 's',
 't',
 'p',
 'a',
 'l',
 'r',
 'v',
 'e',
 'u',
 ' ',
 'o',
 'w',
 'i',
 'n',
 'd',
 'c',
 'y',
 's',
 't',
 'h',
 'l',
 'j',
 'n',
 'd',
 'g',
 'h',
 'm',
 'a',
 'e',
 'u',
 'i',
 'o',
 'c',
 'y',
 ' ',
 'b',
 'r',
 's',
 'f',
 't',
 'l',
 'w',
 'n',
 'd',
 'g',
 'h',
 'v',
 'k',
 'm',
 'a',
 'e',
 'u',
 'i',
 'o',
 'c',
 'y',
 ' ',
 'b',
 'r',
 's',
 'f',
 't',
 'l',
 'w',
 'n',
 'd',
 'g',
 'h',
 'v',
 'm',
 'a',
 'e',
 'i',
 'o',
 'c',
 'y',
 ' ',
 'b',
 'r',
 's',
 'f',
 't',
 'a',
 'p',
 'r',
 'v',
 'e',
 ' ',
 'o',
 'i',
 'w',
 'n',
 'd',
 'm',
 'c',
 's',
 't',
 'h',
 'a',
 'p',
 'r',
 'e',
 'u',
 'i',
 'w',
 'o',
 'n',
 'd',
 'g',
 'c',
 'm',
 'h',
 's',
 'f',
 't',
 ' ',
 'l',
 'a',
 'p',
 'r',
 'e',
 'i',
 'o',
 'w',
 'n',
 'm',
 'd',
 'c',
 'y',
 'h',
 's',
 't',
 ' ',
 'l',
 'w',
 'n',
 'd',
 'h',
 'm',
 'a',
 'e',
 'u',
 'i',
 'o',
 'c',
 'y',
 ' ',
 'b'

This converts vocab_list into a set to remove any duplicate characters and then back into a list.

In [12]:
vocab_list = list(set(vocab_list))

In [14]:
vocab_list

['l',
 'x',
 ',',
 'j',
 'w',
 'n',
 'd',
 'g',
 'h',
 'v',
 'z',
 'k',
 'm',
 'a',
 'e',
 '?',
 'u',
 'i',
 'o',
 'c',
 'y',
 ' ',
 'b',
 'r',
 ';',
 'q',
 '.',
 ':',
 "'",
 's',
 'f',
 't',
 'p']

This creates a dictionary vocab_dict where each unique character is mapped to a unique integer ID as tokens, starting from 0.

In [13]:
vocab_dict = {v: k for k, v in enumerate(vocab_list)}

This merges the mapping for the space character " " with the pipe character "|". The space character is deleted from the dictionary after its ID is reassigned to the pipe character.   

To make it clearer that " " has its own token class, we give it a more visible character "|". In addition, we also add an "[UNK]" token so that the model can later deal with characters not encountered in the training set.

In [15]:
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

In [16]:
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)
len(vocab_dict)

35

In [17]:
vocab_dict

{'l': 0,
 'x': 1,
 ',': 2,
 'j': 3,
 'w': 4,
 'n': 5,
 'd': 6,
 'g': 7,
 'h': 8,
 'v': 9,
 'z': 10,
 'k': 11,
 'm': 12,
 'a': 13,
 'e': 14,
 '?': 15,
 'u': 16,
 'i': 17,
 'o': 18,
 'c': 19,
 'y': 20,
 'b': 22,
 'r': 23,
 ';': 24,
 'q': 25,
 '.': 26,
 ':': 27,
 "'": 28,
 's': 29,
 'f': 30,
 't': 31,
 'p': 32,
 '|': 21,
 '[UNK]': 33,
 '[PAD]': 34}

In [18]:
import json
with open('vocab.json', 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)

## Tokenizer

A `Wav2Vec2CTCTokenizer` is created using the saved vocab.json file.  

It defines the following special tokens:  
unk_token: "[UNK]" for unknown characters.  
pad_token: "[PAD]" for padding.  
word_delimiter_token: "|", which is used as the token for separating words (since spaces were replaced by pipes earlier).   

[🤗 Doc](https://huggingface.co/docs/transformers/en/model_doc/wav2vec2#transformers.Wav2Vec2CTCTokenizer) for Wav2Vec2 Tokenizer.

In [19]:
tokenizer = Wav2Vec2CTCTokenizer("./vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|")

## Data Prepartion: Feature Extractor

The feature extractor **standardizes raw audio into normalized, padded tensors** so Wav2Vec2 can learn meaningful speech representations.

The `Wav2Vec2FeatureExtractor` is initialized to process audio features.

Key parameters:  
feature_size=1: Indicates 1D audio features.  
sampling_rate=16000: Audio is expected to have a sampling rate of 16,000 Hz.   
padding_value=0.0: Padding values are set to 0.0.  
do_normalize=True: Normalization of audio is enabled.  
return_attention_mask=False: Attention masks are not used.  

[🤗 Docs](https://huggingface.co/docs/transformers/en/model_doc/wav2vec2#transformers.Wav2Vec2FeatureExtractor) for Wav2Vec2FeatureExtractor

In [21]:
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=False)

`Wav2Vec2Processor` combines feature extractor and tokenizer.  

[🤗 Docs](https://huggingface.co/docs/transformers/en/model_doc/wav2vec2#transformers.Wav2Vec2Processor) for Wav2Vec2Processor

In [22]:
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

In [23]:
print("Target text:", dataset["train"][56]["text"])
print("Input array shape:", np.asarray(dataset["train"][56]["audio"]["array"]).shape)
print("Sampling rate:", dataset["train"][56]["audio"]["sampling_rate"])

Target text: the first location within my training data set now in order to do it for training i would
Input array shape: (76624,)
Sampling rate: 16000


This function prepares the data for training by extracting features.

In [24]:
def prepare_dataset(batch):
    audio = batch["audio"]
    # batched output is "un-batched" to ensure mapping is correct
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

In [25]:
dataset = dataset.map(prepare_dataset, batch_size=8)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

In [26]:
dataset = dataset['train'].train_test_split(test_size=0.1, shuffle=True, seed=42)

## Data Collator

The `DataCollatorCTCWithPadding` dynamically pads input audio and label sequences to equal lengths within a batch. It uses the processor for audio padding and tokenizer for labels, then replaces padded label values with -100 so they are ignored during CTC loss computation, ensuring efficient batching and correct model training.


In [27]:
@dataclass
class DataCollatorCTCWithPadding:
    """
    Data collator that will dynamically pad the inputs received.
    Args:
        processor (:class:`~transformers.Wav2Vec2Processor`)
            The processor used for proccessing the data.
        padding (:obj:`bool`, :obj:`str` or :class:`~transformers.tokenization_utils_base.PaddingStrategy`, `optional`, defaults to :obj:`True`):
            Select a strategy to pad the returned sequences (according to the model's padding side and padding index)
            among:
            * :obj:`True` or :obj:`'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
              sequence if provided).
            * :obj:`'max_length'`: Pad to a maximum length specified with the argument :obj:`max_length` or to the
              maximum acceptable input length for the model if that argument is not provided.
            * :obj:`False` or :obj:`'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of
              different lengths).
    """

    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lenghts and need
        # different padding methods
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        # Use tokenizer's pad method directly for labels
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels

        return batch

In [28]:
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

## Evaluation Metric

In [29]:
wer_metric = evaluate.load("wer")

In [30]:
def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    # we do not want to group tokens when computing the metrics
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

## Wav2Vec2 Model + CTC head

Base Pretrained model is defined and parameters are set for training.  

Refer to [🤗 Trainer](https://huggingface.co/docs/transformers/en/main_classes/trainer) and [🤗 TrainingArguments](https://huggingface.co/docs/transformers/v4.44.2/en/main_classes/trainer#transformers.TrainingArguments) to learn about the Parameters.

---

`AutoModelForCTC` is a **Hugging Face class that automatically loads a speech recognition model designed for CTC (Connectionist Temporal Classification)**.

### What is CTC (Connectionist Temporal Classification)?

CTC is a technique used in speech recognition where:

* Input = long audio signal
* Output = shorter text (transcript)
* No exact alignment between audio frames and characters is needed

👉 The model learns:

> “Which parts of audio correspond to which characters” automatically

---

### 🔧 What `AutoModelForCTC` does

On running:

```python
model = AutoModelForCTC.from_pretrained("facebook/wav2vec2-base")
```

It:

#### 1. Loads the correct architecture

* In this case → **Wav2Vec2 + CTC head**
* Adds a classification layer on top of the base model

---

#### 2. Adds a prediction layer (important)

* Converts hidden features → vocabulary logits
* Output shape:

  ```
  (batch_size, time_steps, vocab_size)
  ```

---

#### 3. Supports CTC loss for training

* Automatically computes loss if you pass labels

---

### 🔄 Pipeline connection

Your full pipeline is:

```
Audio → FeatureExtractor → Wav2Vec2 → CTC Layer → Text
```

* Feature extractor → cleans audio
* Model (`AutoModelForCTC`) → predicts characters

---

👉 **CTC head converts audio features into text predictions without needing aligned labels**.


Wav2Vec uses CNN-based self-supervised learning for audio features, while Wav2Vec 2.0 improves it with Transformers and masking, enabling high-accuracy end-to-end speech recognition with less labeled data.

In [31]:
model = AutoModelForCTC.from_pretrained(
    "facebook/wav2vec2-base",
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer)
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [32]:
model.freeze_feature_encoder()

In [33]:
training_args = TrainingArguments(
  output_dir="AsrTaskModel",
  group_by_length=False,
  per_device_train_batch_size=8,
  per_device_eval_batch_size=8,
  eval_strategy="steps",
  num_train_epochs=7,
  fp16=True,
  gradient_checkpointing=True,
  save_steps=1000,
  eval_steps=500,
  logging_steps=500,
  learning_rate=1e-4,
  weight_decay=0.005,
  warmup_steps=1000,
  save_total_limit=2,
  load_best_model_at_end=True,
  save_strategy="steps"
)

In [34]:
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
    # Removed model_input_names as it's not supported in this Trainer version
)

In [35]:
trainer.train()

Step,Training Loss,Validation Loss,Wer
500,7.475037,2.949114,0.999742
1000,2.917841,2.492158,1.000000
1500,1.223443,0.735178,0.470143
2000,0.735976,0.565588,0.375290
2500,0.592266,0.477193,0.329496
3000,0.482375,0.455275,0.305267
3500,0.431372,0.434447,0.284732
4000,0.356599,0.435703,0.274250
4500,0.329759,0.411602,0.265659
5000,0.278688,0.422992,0.256036


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6300, training_loss=1.2258319007025824, metrics={'train_runtime': 6604.8642, 'train_samples_per_second': 7.631, 'train_steps_per_second': 0.954, 'total_flos': 4.931067815680836e+18, 'train_loss': 1.2258319007025824, 'epoch': 7.0})